# EDA (TV1, GĐ2)\nChỉ chạy khi TV2 đã cung cấp `data/processed/it_salary_clean.csv`. Không sinh biểu đồ giả khi thiếu trường.\n

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
PATH=ROOT/'data/processed/it_salary_clean.csv'
if not PATH.exists(): raise FileNotFoundError('Cần TV2 tạo it_salary_clean.csv trước khi EDA')
df=pd.read_csv(PATH)
print('Rows:',len(df),'Columns:',df.columns.tolist())
TARGET=next((c for c in ['salary_label','salary_target','salary_million','salary_mid'] if c in df.columns and pd.to_numeric(df[c],errors='coerce').notna().sum()>0),None)
if TARGET is None:raise ValueError('Thiếu nhãn lương dạng số. Yêu cầu TV2 xác nhận schema.')
df['_y']=pd.to_numeric(df[TARGET],errors='coerce')
df=df[df['_y'].notna() & (df['_y']>0)].copy()
OUT=ROOT/'reports/figures';OUT.mkdir(parents=True,exist_ok=True)
log=[f'rows={len(df)}',f'target={TARGET}',df['_y'].describe().to_string()]
def save(name):
 plt.tight_layout();plt.savefig(OUT/('gd2_'+name+'.png'),dpi=180,bbox_inches='tight');plt.show();plt.close()
def col(*names):
 for n in names:
  if n in df.columns:return n
 raise KeyError('Missing EDA dimension, ask TV2: '+', '.join(names))


In [ ]:
plt.figure(figsize=(8,4));plt.hist(df['_y'],bins=25);plt.xlabel('Triệu VNĐ/tháng (xác nhận đơn vị với TV2)');plt.ylabel('Số tin');plt.title('Phân phối lương');save('01_salary_hist')

In [ ]:
plt.figure(figsize=(8,3));plt.boxplot(df['_y'],vert=False);plt.xlabel('Triệu VNĐ/tháng');plt.title('Ngoại lệ lương');save('02_salary_box')

In [ ]:
c=col(*['seniority', 'level', 'seniority_group']);g=df.groupby(c)['_y'].median().sort_values(ascending=False).head(12);plt.figure(figsize=(9,4));g.plot.bar();plt.ylabel('Median triệu VNĐ/tháng');plt.title('Lương theo '+c);plt.xticks(rotation=35,ha='right');save('03_seniority')

In [ ]:
c=col(*['city', 'city_group']);g=df.groupby(c)['_y'].median().sort_values(ascending=False).head(12);plt.figure(figsize=(9,4));g.plot.bar();plt.ylabel('Median triệu VNĐ/tháng');plt.title('Lương theo '+c);plt.xticks(rotation=35,ha='right');save('04_city')

In [ ]:
c=col(*['job_group', 'job_family', 'it_role_type']);g=df.groupby(c)['_y'].median().sort_values(ascending=False).head(12);plt.figure(figsize=(9,4));g.plot.bar();plt.ylabel('Median triệu VNĐ/tháng');plt.title('Lương theo '+c);plt.xticks(rotation=35,ha='right');save('05_job_group')

In [ ]:
c=col(*['experience_group', 'exp_group', 'experience_years']);g=df.groupby(c)['_y'].median().sort_values(ascending=False).head(12);plt.figure(figsize=(9,4));g.plot.bar();plt.ylabel('Median triệu VNĐ/tháng');plt.title('Lương theo '+c);plt.xticks(rotation=35,ha='right');save('06_experience')

In [ ]:
c=col('skills','technical_skills','skills_normalized')
counter=Counter()
for raw in df[c].dropna().astype(str):
 for t in raw.replace(';',',').split(','):
  t=t.strip().lower()
  if t:counter[t]+=1
if not counter:raise ValueError('Không có kỹ năng hợp lệ')
plt.figure(figsize=(10,4));pd.Series(dict(counter.most_common(12))).plot.bar();plt.ylabel('Số tin');plt.title('Top kỹ năng (tách sơ bộ theo dấu phẩy)');plt.xticks(rotation=35,ha='right');save('07_skills')
(ROOT/'reports/gd2_eda_output.txt').write_text('\n'.join(log),encoding='utf-8');print('Saved 7 figures and EDA summary')
